In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import pathlib
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import copy
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================
# 1. TÍNH TÁI LẬP (Reproducibility)
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ==========================================
# 2. CẤU HÌNH HỆ THỐNG
# ==========================================
GroupID = "04" 
IMAGE_SIZE = 224 # [cite: 472-476]
BATCH_SIZE = 32      
EPOCHS = 60 
MAX_LR = 1.5e-3 

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ==========================================
# 3. CHUẨN BỊ DỮ LIỆU & RANDOM ERASING
# ==========================================
data_dir = pathlib.Path("/kaggle/input/datasets/huynhthethien/radarcommunsignaldata2026train")

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.15), value='random')
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = datasets.ImageFolder(root=str(data_dir), transform=train_transform)
num_classes = len(dataset.classes)
class_names = dataset.classes

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

dataset_val = copy.deepcopy(dataset)
dataset_val.transform = val_transform
val_dataset.dataset = dataset_val

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ==========================================
# 4. KIẾN TRÚC MÔ HÌNH (~98.6k Params, MAX CAPACITY)
# ==========================================
class SEModule(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        # Đảm bảo mid_channels luôn >= 1
        mid_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid_channels, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(mid_channels, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class InvertedResidual(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio):
        super().__init__()
        hidden_dim = int(inp * expand_ratio)
        self.use_res_connect = stride == 1 and inp == oup

        layers = []
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(inp, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True)
            ])
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True),
            SEModule(hidden_dim, reduction=16), # Giữ mức nén 16 để dồn tham số cho Convolution
            nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
            nn.BatchNorm2d(oup),
        ])
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        return self.conv(x)

class RFMobileNetV2(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # KERNEL 5x5: Lấy khung nhìn rộng ngay từ đầu
        self.stem = nn.Sequential(
            nn.Conv2d(3, 24, 5, 2, 2, bias=False), 
            nn.BatchNorm2d(24),
            nn.SiLU(inplace=True)
        )
        self.blocks = nn.Sequential(
            InvertedResidual(24, 24, 1, 1),
            InvertedResidual(24, 32, 2, 4),
            InvertedResidual(32, 48, 2, 4),
            InvertedResidual(48, 64, 2, 4), # Tăng expand_ratio lên 4
            InvertedResidual(64, 80, 2, 2.5), # Kênh 80, exp 2.5 để khít tham số
        )
        self.head = nn.Sequential(
            nn.Conv2d(80, 128, 1, 1, 0, bias=False), 
            nn.BatchNorm2d(128),
            nn.SiLU(inplace=True),
            nn.AdaptiveAvgPool2d((2, 2)) 
        )
        self.dropout = nn.Dropout(0.4) 
        self.classifier = nn.Linear(128 * 2 * 2, num_classes) # Raw logits [cite: 501-502]

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.classifier(x)

model = RFMobileNetV2(num_classes).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*60)
print("THÔNG TIN CẤU HÌNH [ULTIMATE VERSION]")
print("="*60)
print(f"Device:              {DEVICE}")
print(f"Tổng số tham số:     {total_params} (Mục tiêu đồ án: < 100,000)")
print("Kỹ thuật:            Kernel 5x5, Random Erasing, Pool 2x2, 100 Epochs")
print("="*60)

# ==========================================
# 5. THIẾT LẬP HUẤN LUYỆN
# ==========================================
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 
optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=1e-2)

# Long-tail OneCycleLR: Chỉ 20% warmup, 80% thời gian để tinh chỉnh trọng số
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, steps_per_epoch=len(train_loader), epochs=EPOCHS, pct_start=0.2
)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def train_model():
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(EPOCHS):
        model.train()
        r_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            r_loss += loss.item() * images.size(0)
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
            
        train_loss = r_loss / len(train_dataset)
        train_acc = correct / total

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                v_loss += loss.item() * images.size(0)
                _, pred = torch.max(outputs, 1)
                v_total += labels.size(0)
                v_correct += (pred == labels).sum().item()
        
        val_loss = v_loss / len(val_dataset)
        val_acc = v_correct / v_total
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f"Epoch [{epoch+1:03d}/{EPOCHS}] | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} --> LƯU KỶ LỤC MỚI!")
        else:
            print(f"Epoch [{epoch+1:03d}/{EPOCHS}] | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
            
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

    model.load_state_dict(best_model_wts)
    return best_val_acc

# ==========================================
# 6. THỰC THI VÀ ĐÁNH GIÁ
# ==========================================
print("\n[INFO] Bắt đầu quá trình huấn luyện cường độ cao...")
best_acc = train_model()
print(f"\n[INFO] Độ chính xác cao nhất trên tập Validation: {best_acc:.4f}")

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS+1), history['train_loss'], label='Train Loss', marker='o', markersize=3)
plt.plot(range(1, EPOCHS+1), history['val_loss'], label='Val Loss', marker='s', markersize=3)
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS+1), history['train_acc'], label='Train Acc', marker='o', markersize=3)
plt.plot(range(1, EPOCHS+1), history['val_acc'], label='Val Acc', marker='s', markersize=3)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(f'{GroupID}_Training_Metrics.png', dpi=300, bbox_inches='tight')
plt.close()

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, pred = torch.max(outputs, 1)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n--- BÁO CÁO PHÂN LOẠI (CLASSIFICATION REPORT) ---")
print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix on Validation Set')
plt.savefig(f'{GroupID}_Confusion_Matrix.png', dpi=300, bbox_inches='tight')
plt.close()

# ==========================================
# 7. XUẤT MÔ HÌNH (KHÔNG ĐƯỢC CHỈNH SỬA) [cite: 516-527]
# ==========================================
############################################
# DO NOT MODIFY THIS SECTION
############################################
model.eval()
example_input = torch.randn(1,3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
traced_model = torch.jit.trace(model, example_input)

model_name = f"{GroupID}_DeepLearningProject_TrainedModel.pt"
traced_model.save(model_name)
print("\n[INFO] Model saved:", model_name)

THÔNG TIN CẤU HÌNH [ULTIMATE VERSION]
Device:              cuda:0
Tổng số tham số:     98540 (Mục tiêu đồ án: < 100,000)
Kỹ thuật:            Kernel 5x5, Random Erasing, Pool 2x2, 100 Epochs

[INFO] Bắt đầu quá trình huấn luyện cường độ cao...
Epoch [001/60] | Train Acc: 0.4386 | Val Acc: 0.6406 --> LƯU KỶ LỤC MỚI!
Epoch [002/60] | Train Acc: 0.6841 | Val Acc: 0.7560 --> LƯU KỶ LỤC MỚI!
Epoch [003/60] | Train Acc: 0.7645 | Val Acc: 0.8150 --> LƯU KỶ LỤC MỚI!
Epoch [004/60] | Train Acc: 0.8054 | Val Acc: 0.8387 --> LƯU KỶ LỤC MỚI!
Epoch [005/60] | Train Acc: 0.8271 | Val Acc: 0.8442 --> LƯU KỶ LỤC MỚI!
Epoch [006/60] | Train Acc: 0.8413 | Val Acc: 0.8632 --> LƯU KỶ LỤC MỚI!
Epoch [007/60] | Train Acc: 0.8505 | Val Acc: 0.8678 --> LƯU KỶ LỤC MỚI!
Epoch [008/60] | Train Acc: 0.8574 | Val Acc: 0.8320
Epoch [009/60] | Train Acc: 0.8629 | Val Acc: 0.8686 --> LƯU KỶ LỤC MỚI!
Epoch [010/60] | Train Acc: 0.8657 | Val Acc: 0.8760 --> LƯU KỶ LỤC MỚI!
Epoch [011/60] | Train Acc: 0.8703 | Val Acc: 